# Using the `analogues` module

First, make sure your environment (here assumed to be named `.venv`) is up to date by running:
```bash
source .venv/bin/activate
pip install .
```

The setup below finds the repo root automatically, imports the explicit classes used in this notebook, resolves the TNG outputs path without relying on a colleague-specific hardcoded location, and loads the analogue selection and filter settings from `config/config.json`. You can override the automatic path discovery with `TNG_BASE_PATH` or `ILLUSTRIS_BASE_PATH`.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_root = next((path for path in [cwd, *cwd.parents] if (path / "analogues").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repo root containing the `analogues` package.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analogues import AnalogueSample, FilterConfig, SelectionConfig
from config.loader import load_config

print("Repo root:", repo_root)


This gives access to the main public classes used here. The most straightforward way to produce the analogue sample is to use the `AnalogueSample` class. It expects a `base_path` pointing to the TNG outputs directory, a `selection_config` controlling the initial subhalo sample, and a `filter_config` controlling the pair-level cuts. The next cell resolves `base_path` automatically, loads the shared config from `config/config.json`, and then builds the sample.


In [ ]:
def resolve_base_path(repo_root: Path) -> str:
    raw_candidates = [
        os.environ.get("TNG_BASE_PATH"),
        os.environ.get("ILLUSTRIS_BASE_PATH"),
        str(repo_root / "tng300" / "outputs"),
    ]

    for raw in raw_candidates:
        if not raw:
            continue
        base = Path(raw).expanduser()
        for candidate in (base, base / "tng300" / "outputs"):
            if (candidate / "groups_099").exists():
                return str(candidate.resolve())

    raise FileNotFoundError(
        "Could not find a valid TNG outputs directory. Set TNG_BASE_PATH or ILLUSTRIS_BASE_PATH "
        "to the outputs directory, or to a root that contains tng300/outputs."
    )


config_loader = load_config()
analysis_config = config_loader.get_config()

base_path = resolve_base_path(repo_root)
snap = 99
print("Using base_path:", base_path)

selection_config = SelectionConfig(**analysis_config.selection)
filter_config = FilterConfig(**analysis_config.filter)

print("Selection config loaded from config/config.json:")
display(pd.DataFrame([analysis_config.selection]))
print("Filter config loaded from config/config.json:")
display(pd.DataFrame([analysis_config.filter]))
print("Active features in shared config:", list(analysis_config.features))
print("Active target in shared config:", analysis_config.target_name)

analogue_sample = AnalogueSample(
    base_path=base_path,
    selection_config=selection_config,
    filter_config=filter_config,
    verbose=True,
    snap=snap,
)


Finally the analogue pairs can be accessed via the `pairs` attribute:


In [ ]:
pairs = analogue_sample.pairs
